In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels'
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import Counter
import random
from tqdm import tqdm

random.seed(42)

print("Libraries imported")

Libraries imported


In [ ]:
def safe_save_path(path):
    """Ensure directory exists in Google Drive before use"""
    os.makedirs(path, exist_ok=True)
    return path

# Paths
PROCESSED_DATASET = safe_save_path(os.path.join(PROJECT_ROOT, 'processed_dataset'))
VALIDATION_DIR = safe_save_path(os.path.join(PROJECT_ROOT, 'validation_samples'))
RESULTS_DIR = safe_save_path(os.path.join(PROJECT_ROOT, 'analysis_results'))

# Load severity thresholds
threshold_file = os.path.join(RESULTS_DIR, 'severity_thresholds.json')
with open(threshold_file, 'r') as f:
    SEVERITY_THRESHOLDS = json.load(f)

SEVERITY_THRESHOLDS = {int(k): v for k, v in SEVERITY_THRESHOLDS.items()}

# Class config
CLASS_NAMES = {
    0: 'Longitudinal Crack',
    1: 'Transverse Crack',
    2: 'Alligator Crack',
    4: 'Pothole'
}

USED_CLASSES = [0, 1, 2, 4]
SEVERITY_LABELS = ['Rendah', 'Sedang', 'Tinggi']

print("Configuration loaded")
print(f"Processed dataset: {PROCESSED_DATASET}")
print(f"Validation output: {VALIDATION_DIR}")

Configuration loaded
Processed dataset: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/processed_dataset
Validation output: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples


In [ ]:
def load_severity_annotation(json_path):
    """Load severity annotation from JSON file"""
    with open(json_path, 'r') as f:
        return json.load(f)

def parse_yolo_label(label_path):
    """Parse YOLO label file"""
    bboxes = []
    if not os.path.exists(label_path):
        return bboxes

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            bboxes.append({
                'class_id': int(parts[0]),
                'x_center': float(parts[1]),
                'y_center': float(parts[2]),
                'width': float(parts[3]),
                'height': float(parts[4])
            })

    return bboxes

print("Helper functions defined")

Helper functions defined


In [ ]:
print("Selecting validation samples...")

# Sample 50 images per class (200 total)
SAMPLES_PER_CLASS = 50

train_severity_dir = os.path.join(PROCESSED_DATASET, 'train', 'severity')
all_severity_files = [f for f in os.listdir(train_severity_dir) if f.endswith('.json')]

# Organize by class
images_by_class = {c: [] for c in USED_CLASSES}

for sev_file in all_severity_files:
    sev_path = os.path.join(train_severity_dir, sev_file)
    data = load_severity_annotation(sev_path)

    # Get classes in this image
    classes_in_image = set([ann['class_id'] for ann in data['annotations']])

    for class_id in classes_in_image:
        if class_id in USED_CLASSES:
            images_by_class[class_id].append(data['image_name'])

print("\nAvailable images per class:")
for class_id in USED_CLASSES:
    print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): {len(images_by_class[class_id])} images")

# Sample stratified
selected_samples = {}
for class_id in USED_CLASSES:
    available = list(set(images_by_class[class_id]))  # Remove duplicates
    n_samples = min(SAMPLES_PER_CLASS, len(available))
    selected_samples[class_id] = random.sample(available, n_samples)
    print(f"  Selected {n_samples} samples for class {class_id}")

# Flatten to unique list
all_selected = set()
for samples in selected_samples.values():
    all_selected.update(samples)

all_selected = list(all_selected)
print(f"\nTotal unique images selected for validation: {len(all_selected)}")

Selecting validation samples...

Available images per class:
  Class 0 (Longitudinal Crack): 5202 images
  Class 1 (Transverse Crack): 3284 images
  Class 2 (Alligator Crack): 3362 images
  Class 4 (Pothole): 1574 images
  Selected 50 samples for class 0
  Selected 50 samples for class 1
  Selected 50 samples for class 2
  Selected 50 samples for class 4

Total unique images selected for validation: 200


In [ ]:
def visualize_validation_sample(img_name, save_dir):
    """Create visualization with bbox and severity labels"""

    # Paths
    img_path = os.path.join(PROCESSED_DATASET, 'train', 'images', img_name)
    lbl_path = os.path.join(PROCESSED_DATASET, 'train', 'labels',
                           os.path.splitext(img_name)[0] + '.txt')
    sev_path = os.path.join(PROCESSED_DATASET, 'train', 'severity',
                           os.path.splitext(img_name)[0] + '.json')

    # Load data
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    bboxes = parse_yolo_label(lbl_path)
    sev_data = load_severity_annotation(sev_path)

    # Color mapping for severity
    severity_colors = {
        0: (0, 255, 0),      # Rendah = Green
        1: (255, 165, 0),    # Sedang = Orange
        2: (255, 0, 0)       # Tinggi = Red
    }

    # Draw bboxes with severity
    for bbox, sev_ann in zip(bboxes, sev_data['annotations']):
        x_center = int(bbox['x_center'] * w)
        y_center = int(bbox['y_center'] * h)
        box_w = int(bbox['width'] * w)
        box_h = int(bbox['height'] * h)

        x1 = x_center - box_w // 2
        y1 = y_center - box_h // 2
        x2 = x_center + box_w // 2
        y2 = y_center + box_h // 2

        severity = sev_ann['severity']
        color = severity_colors[severity]

        # Draw bbox
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

        # Draw label
        label = f"C{bbox['class_id']}-{SEVERITY_LABELS[severity]}"
        label_bg_y1 = max(0, y1 - 25)
        cv2.rectangle(img, (x1, label_bg_y1), (x1 + 150, y1), color, -1)
        cv2.putText(img, label, (x1 + 5, y1 - 8),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Save
    output_path = os.path.join(save_dir, img_name)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, img_bgr)

    return sev_data

print("Generating validation visualizations...")

# Create subdirectories per class
for class_id in USED_CLASSES:
    safe_save_path(os.path.join(VALIDATION_DIR, f'class_{class_id}'))

validation_metadata = []

for class_id in USED_CLASSES:
    class_dir = os.path.join(VALIDATION_DIR, f'class_{class_id}')

    print(f"\nProcessing Class {class_id} ({CLASS_NAMES[class_id]})...")

    for img_name in tqdm(selected_samples[class_id]):
        sev_data = visualize_validation_sample(img_name, class_dir)

        # Store metadata
        for ann in sev_data['annotations']:
            if ann['class_id'] == class_id:
                validation_metadata.append({
                    'image_name': img_name,
                    'class_id': class_id,
                    'class_name': CLASS_NAMES[class_id],
                    'severity': ann['severity'],
                    'severity_label': ann['severity_label'],
                    'area_ratio': ann['area_ratio'],
                    'auto_assigned': True,
                    'manual_review': None,  # To be filled manually
                    'notes': ''
                })

print(f"\nVisualization complete")

Generating validation visualizations...

Processing Class 0 (Longitudinal Crack)...


100%|██████████| 50/50 [00:36<00:00,  1.36it/s]



Processing Class 1 (Transverse Crack)...


100%|██████████| 50/50 [00:27<00:00,  1.85it/s]



Processing Class 2 (Alligator Crack)...


100%|██████████| 50/50 [00:25<00:00,  1.93it/s]



Processing Class 4 (Pothole)...


100%|██████████| 50/50 [00:25<00:00,  1.96it/s]


Visualization complete


In [ ]:
# Create DataFrame
df_validation = pd.DataFrame(validation_metadata)

# Add empty columns for manual review
df_validation['manual_severity'] = ''
df_validation['is_correct'] = ''
df_validation['reviewer_notes'] = ''

# Save to CSV
checklist_path = os.path.join(VALIDATION_DIR, 'validation_checklist.csv')
df_validation.to_csv(checklist_path, index=False)

print(f"Validation checklist created:")
print(f"   {checklist_path}")
print(f"\nTotal annotations to review: {len(df_validation)}")

# Show distribution
print("\nDistribution by class and severity:")
dist = df_validation.groupby(['class_id', 'severity_label']).size().unstack(fill_value=0)
print(dist)

Validation checklist created:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/validation_checklist.csv

Total annotations to review: 316

Distribution by class and severity:
severity_label  Rendah  Sedang  Tinggi
class_id                              
0                   50      27      26
1                   15      25      31
2                   16      23      22
4                   29      31      21


In [ ]:
instructions = """
# PANDUAN VALIDASI MANUAL SEVERITY ANNOTATION

## Tujuan
Memvalidasi akurasi threshold severity otomatis dengan cara manual review pada 200 sample images.

## Cara Melakukan Validasi

### 1. Buka Folder Validasi
   Lokasi: {validation_dir}

   Struktur folder:
   - class_0/ → Longitudinal Crack samples
   - class_1/ → Transverse Crack samples
   - class_2/ → Alligator Crack samples
   - class_4/ → Pothole samples

### 2. Review Setiap Gambar

   Setiap gambar memiliki bounding box dengan warna:
   - 🟢 HIJAU = Rendah
   - 🟠 ORANGE = Sedang
   - 🔴 MERAH = Tinggi

   Label format: "C{{class_id}}-{{Severity}}"
   Contoh: "C0-Sedang" = Class 0 (Longitudinal Crack), Severity Sedang

### 3. Isi Validation Checklist CSV

   File: validation_checklist.csv

   Untuk setiap baris, isi kolom berikut:

   - **manual_severity**: Tulis severity menurut Anda (Rendah/Sedang/Tinggi)
   - **is_correct**: Tulis "Yes" jika setuju dengan auto annotation, "No" jika tidak
   - **reviewer_notes**: (Optional) Catatan kenapa berbeda atau ada keraguan

### 4. Kriteria Penilaian Severity (Panduan)

   **RENDAH:**
   - Kerusakan sangat kecil, tidak mengganggu kendaraan
   - Area kerusakan < 5% dari frame
   - Tidak perlu perbaikan segera

   **SEDANG:**
   - Kerusakan terlihat jelas, bisa mengganggu kenyamanan
   - Area kerusakan 5-15% dari frame
   - Perlu dijadwalkan perbaikan dalam beberapa minggu

   **TINGGI:**
   - Kerusakan besar, berbahaya untuk pengendara (terutama motor)
   - Area kerusakan > 15% dari frame
   - Perlu perbaikan segera (prioritas tinggi)

### 5. Setelah Selesai Review

   - Save file validation_checklist.csv
   - Hitung inter-rater agreement:
     * Berapa persen yang "is_correct" = "Yes"?
     * Jika < 70%, threshold perlu disesuaikan

   - Upload hasil review untuk analisis di Notebook selanjutnya

## Catatan Penting
- Review dilakukan oleh minimal 2 orang untuk mengurangi bias
- Jika ragu antara Rendah-Sedang atau Sedang-Tinggi, pilih yang lebih konservatif (lebih tinggi)
- Fokus pada area kerusakan relatif terhadap frame, bukan ukuran absolut

""".format(validation_dir=VALIDATION_DIR)

# Save instructions
instructions_path = os.path.join(VALIDATION_DIR, 'VALIDATION_INSTRUCTIONS.txt')
with open(instructions_path, 'w', encoding='utf-8') as f:
    f.write(instructions)

print("Validation instructions created:")
print(f"   {instructions_path}")
print("\n")
print(instructions)

Validation instructions created:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/VALIDATION_INSTRUCTIONS.txt



# PANDUAN VALIDASI MANUAL SEVERITY ANNOTATION

## Tujuan
Memvalidasi akurasi threshold severity otomatis dengan cara manual review pada 200 sample images.

## Cara Melakukan Validasi

### 1. Buka Folder Validasi
   Lokasi: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples
   
   Struktur folder:
   - class_0/ → Longitudinal Crack samples
   - class_1/ → Transverse Crack samples
   - class_2/ → Alligator Crack samples
   - class_4/ → Pothole samples

### 2. Review Setiap Gambar
   
   Setiap gambar memiliki bounding box dengan warna:
   - 🟢 HIJAU = Rendah
   - 🟠 ORANGE = Sedang
   - 🔴 MERAH = Tinggi
   
   Label format: "C{class_id}-{Severity}"
   Contoh: "C0-Sedang" = Class 0 (Longitudinal Crack), Severity Sedang

### 3. Isi Validation Checklist CSV
   
   File: validation_checklist.csv
   
   Untuk seti

In [ ]:
summary_report = {
    'total_images': len(all_selected),
    'total_annotations': len(validation_metadata),
    'samples_per_class': {int(k): len(v) for k, v in selected_samples.items()},
    'severity_distribution': df_validation.groupby('severity_label').size().to_dict(),
    'validation_checklist_path': checklist_path,
    'instructions_path': instructions_path,
    'validation_images_dir': VALIDATION_DIR
}

# Save summary
summary_path = os.path.join(VALIDATION_DIR, 'validation_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n")
print("VALIDATION SUMMARY")
print("="*60)
print(f"Total images: {summary_report['total_images']}")
print(f"Total annotations: {summary_report['total_annotations']}")
print(f"\nSamples per class:")
for class_id, count in summary_report['samples_per_class'].items():
    print(f"  Class {class_id}: {count} images")
print(f"\nSeverity distribution:")
for sev, count in summary_report['severity_distribution'].items():
    print(f"  {sev}: {count} annotations")

print(f"\nSummary saved to: {summary_path}")



VALIDATION SUMMARY
Total images: 200
Total annotations: 316

Samples per class:
  Class 0: 50 images
  Class 1: 50 images
  Class 2: 50 images
  Class 4: 50 images

Severity distribution:
  Rendah: 110 annotations
  Sedang: 106 annotations
  Tinggi: 100 annotations

Summary saved to: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/validation_summary.json


In [ ]:
print("\n")
print("NOTEBOOK 3 COMPLETE")
print("="*60)
print("\nGenerated files:")
print(f"   {VALIDATION_DIR}/")
print("     ├── class_0/ (Longitudinal Crack samples)")
print("     ├── class_1/ (Transverse Crack samples)")
print("     ├── class_2/ (Alligator Crack samples)")
print("     ├── class_4/ (Pothole samples)")
print("     ├── validation_checklist.csv FILL THIS")
print("     ├── VALIDATION_INSTRUCTIONS.txt")
print("     └── validation_summary.json")



NOTEBOOK 3 COMPLETE

Generated files:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/
     ├── class_0/ (Longitudinal Crack samples)
     ├── class_1/ (Transverse Crack samples)
     ├── class_2/ (Alligator Crack samples)
     ├── class_4/ (Pothole samples)
     ├── validation_checklist.csv FILL THIS
     ├── VALIDATION_INSTRUCTIONS.txt
     └── validation_summary.json


In [ ]:
print("\nSyncing all files to Google Drive...")
from google.colab import drive
drive.flush_and_unmount()
print(" files synced successfully!")
print("If you want to continue working, re-run Cell 1 to mount Drive again.")


Syncing all files to Google Drive...
Drive not mounted, so nothing to flush and unmount.
 files synced successfully!
If you want to continue working, re-run Cell 1 to mount Drive again.
